# Analytical Query Library

This notebook defines and implements the required analytical queries with Spark SQL against the Week 1 Delta tables.

## Scope and common definitions

- **Taxi demand:** number of valid taxi trips, counted with `COUNT(trip_id)`.
- **Taxi zone:** pickup taxi zone; demand is attributed to where a trip starts.
- **Month:** calendar month in the `America/New_York` session time zone.
- **Weather condition:** Meteostat `coco` / integrated `weather_condition_code`, grouped into clear/fair, cloudy, fog, rain, snow/sleet, storm, and unknown.
- **Air quality:** borough-hour mean PM2.5 (`air_quality_pm25_mean`). Demand is first aggregated to the same borough-hour grain so repeated trip-level PM2.5 values do not bias the analysis.
- **Peak hour:** hour(s) with the highest trip count for a day of week; ties are retained.
- Missing weather or air-quality matches are excluded only from analyses that require that context.

The queries return analytical result sets only; they do not write materialized data products.

In [1]:
from pathlib import Path

# This is for Windows Env.
from data_intensive_computing.spark_env import configure_windows_spark_env

configure_windows_spark_env()

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

builder = (
    SparkSession.builder
    .appName("Lab2-Analytical-Queries")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Processed data:", PROCESSED_DIR)

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Processed data: D:\Master\data intensive\data-intensive-computing\data\processed


In [2]:
import time

DELTA_TABLES = {
    "taxi_trips": PROCESSED_DIR / "taxi_trips",
    "weather": PROCESSED_DIR / "weather",
    "air_quality": PROCESSED_DIR / "air_quality",
    "taxi_zones": PROCESSED_DIR / "taxi_zones",
    "integrated_taxi_trips": PROCESSED_DIR / "integrated_taxi_trips",
}

missing = [name for name, path in DELTA_TABLES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing Delta tables: {missing}. Run the Week 1 ingestion and integration notebooks first."
    )

for view_name, path in DELTA_TABLES.items():
    spark.read.format("delta").load(str(path)).createOrReplaceTempView(view_name)
    print(f"Registered {view_name}: {path}")


def run_query(query_name, sql_text, display_rows=50):
    """Execute the complete query, print its elapsed time, and show a preview."""
    started = time.perf_counter()
    result = spark.sql(sql_text)
    rows = result.collect()
    elapsed = time.perf_counter() - started

    print(f"{query_name} execution time: {elapsed:.3f} seconds")
    print(f"{query_name} result rows: {len(rows):,}")
    if rows:
        spark.createDataFrame(rows[:display_rows], schema=result.schema).show(
            display_rows, truncate=False
        )
    return {"rows": rows, "execution_time_seconds": elapsed}


QUERIES = {}
RESULTS = {}

Registered taxi_trips: D:\Master\data intensive\data-intensive-computing\data\processed\taxi_trips
Registered weather: D:\Master\data intensive\data-intensive-computing\data\processed\weather
Registered air_quality: D:\Master\data intensive\data-intensive-computing\data\processed\air_quality
Registered taxi_zones: D:\Master\data intensive\data-intensive-computing\data\processed\taxi_zones
Registered integrated_taxi_trips: D:\Master\data intensive\data-intensive-computing\data\processed\integrated_taxi_trips


## Query 1: Monthly taxi demand for each pickup zone

- Grain: pickup year × pickup month × pickup zone.
- Measure: trip count; zone attributes come from the lookup Delta table.
- `pickup_year` and `pickup_month` remain separate so later experiments can filter their physical partitions directly.

In [3]:
QUERIES["monthly_zone_demand"] = """
SELECT
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown') AS pickup_borough,
    COALESCE(z.zone, 'Unknown') AS pickup_zone,
    COUNT(t.trip_id) AS trip_count
FROM taxi_trips AS t
LEFT JOIN taxi_zones AS z
    ON t.pickup_location_id = z.location_id
GROUP BY
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown'),
    COALESCE(z.zone, 'Unknown')
ORDER BY t.pickup_year, t.pickup_month, trip_count DESC, t.pickup_location_id
"""

RESULTS["monthly_zone_demand"] = run_query(
    "monthly_zone_demand", QUERIES["monthly_zone_demand"]
)

monthly_zone_demand execution time: 101.717 seconds
monthly_zone_demand result rows: 793
+-----------+------------+------------------+--------------+-----------------------------+----------+
|pickup_year|pickup_month|pickup_location_id|pickup_borough|pickup_zone                  |trip_count|
+-----------+------------+------------------+--------------+-----------------------------+----------+
|2002       |12          |50                |Manhattan     |Clinton West                 |1         |
|2002       |12          |132               |Queens        |JFK Airport                  |1         |
|2002       |12          |170               |Manhattan     |Murray Hill                  |1         |
|2008       |12          |141               |Manhattan     |Lenox Hill West              |1         |
|2009       |1           |79                |Manhattan     |East Village                 |1         |
|2009       |1           |137               |Manhattan     |Kips Bay                     |1    

## Query 2: Average trip distance under different weather conditions

- Grain: weather category.
- Measures: average distance, population standard deviation, and trip count.
- Only exact pickup-hour weather matches are included; dispersion and sample size provide context for the mean.

In [4]:
QUERIES["distance_by_weather"] = """
WITH classified AS (
    SELECT
        trip_id,
        trip_distance,
        CASE
            WHEN weather_condition_code IN (1, 2) THEN 'clear_or_fair'
            WHEN weather_condition_code IN (3, 4) THEN 'cloudy'
            WHEN weather_condition_code IN (5, 6) THEN 'fog'
            WHEN weather_condition_code IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN weather_condition_code IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN weather_condition_code BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM integrated_taxi_trips
    WHERE weather_match_status = 'exact_hour'
      AND trip_distance IS NOT NULL
)
SELECT
    weather_condition,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS average_trip_distance,
    ROUND(STDDEV_POP(trip_distance), 3) AS trip_distance_stddev
FROM classified
GROUP BY weather_condition
ORDER BY average_trip_distance DESC
"""

RESULTS["distance_by_weather"] = run_query(
    "distance_by_weather", QUERIES["distance_by_weather"]
)

distance_by_weather execution time: 22.686 seconds
distance_by_weather result rows: 5
+-----------------+----------+---------------------+--------------------+
|weather_condition|trip_count|average_trip_distance|trip_distance_stddev|
+-----------------+----------+---------------------+--------------------+
|clear_or_fair    |3261811   |4.293                |290.636             |
|rain             |1284016   |4.035                |270.101             |
|cloudy           |4337999   |3.95                 |259.113             |
|fog              |263672    |3.517                |157.222             |
|snow_or_sleet    |270173    |3.222                |70.042              |
+-----------------+----------+---------------------+--------------------+



## Query 3: Relationship between air quality and taxi demand

- Grain: borough-hour, matching the grain of PM2.5 observations.
- Measures: average hourly demand by PM2.5 band and Pearson correlation.
- Aggregating before analysis avoids overweighting PM2.5 hours that already have many trips. Only exact borough-hour matches are included; the result describes association, not causation.

In [5]:
QUERIES["air_quality_demand_by_band"] = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
), banded AS (
    SELECT
        *,
        CASE
            WHEN pm25 <= 5.0 THEN '01_very_low_0_5'
            WHEN pm25 <= 15.0 THEN '02_low_5_15'
            WHEN pm25 <= 35.0 THEN '03_elevated_15_35'
            ELSE '04_high_over_35'
        END AS pm25_band
    FROM borough_hour
)
SELECT
    pm25_band,
    COUNT(*) AS borough_hour_count,
    ROUND(AVG(pm25), 3) AS average_pm25,
    ROUND(AVG(hourly_trip_count), 2) AS average_hourly_demand,
    MIN(hourly_trip_count) AS minimum_hourly_demand,
    MAX(hourly_trip_count) AS maximum_hourly_demand
FROM banded
GROUP BY pm25_band
ORDER BY pm25_band
"""

QUERIES["air_quality_demand_correlation"] = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
)
SELECT
    COALESCE(pickup_borough, 'All boroughs') AS pickup_borough,
    COUNT(*) AS matched_borough_hours,
    ROUND(CORR(pm25, hourly_trip_count), 4) AS pm25_demand_pearson_correlation
FROM borough_hour
GROUP BY GROUPING SETS ((pickup_borough), ())
ORDER BY pickup_borough
"""

RESULTS["air_quality_demand_by_band"] = run_query(
    "air_quality_demand_by_band", QUERIES["air_quality_demand_by_band"]
)
RESULTS["air_quality_demand_correlation"] = run_query(
    "air_quality_demand_correlation", QUERIES["air_quality_demand_correlation"]
)

air_quality_demand_by_band execution time: 11.322 seconds
air_quality_demand_by_band result rows: 4
+-----------------+------------------+------------+---------------------+---------------------+---------------------+
|pm25_band        |borough_hour_count|average_pm25|average_hourly_demand|minimum_hourly_demand|maximum_hourly_demand|
+-----------------+------------------+------------+---------------------+---------------------+---------------------+
|01_very_low_0_5  |2917              |3.427       |138.6                |1                    |1047                 |
|02_low_5_15      |2875              |8.265       |152.18               |1                    |911                  |
|03_elevated_15_35|637               |19.938      |146.53               |1                    |809                  |
|04_high_over_35  |6                 |38.1        |289.33               |31                   |584                  |
+-----------------+------------------+------------+---------------------+-

## Query 4: Zones with the largest weather-related demand variation

- Grain: pickup zone × weather category, based on average hourly demand.
- Measures: demand range and population standard deviation across weather categories.
- The query constructs every zone × observed weather-hour combination so zero-trip hours are retained. Hourly normalization avoids favoring weather categories merely because they occur more often.

In [6]:
QUERIES["zone_weather_demand_variation"] = """
WITH classified_weather_hours AS (
    SELECT
        observation_timestamp AS weather_hour,
        CASE
            WHEN coco IN (1, 2) THEN 'clear_or_fair'
            WHEN coco IN (3, 4) THEN 'cloudy'
            WHEN coco IN (5, 6) THEN 'fog'
            WHEN coco IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN coco IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN coco BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM weather
    WHERE observation_timestamp IS NOT NULL
), trip_bounds AS (
    SELECT MIN(pickup_hour) AS first_hour, MAX(pickup_hour) AS last_hour
    FROM taxi_trips
), relevant_weather_hours AS (
    SELECT w.weather_hour, w.weather_condition
    FROM classified_weather_hours AS w
    CROSS JOIN trip_bounds AS b
    WHERE w.weather_hour BETWEEN b.first_hour AND b.last_hour
      AND w.weather_condition <> 'unknown'
), zone_hour_counts AS (
    SELECT
        pickup_location_id AS location_id,
        pickup_hour,
        COUNT(trip_id) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_location_id, pickup_hour
), zone_weather_hour_grid AS (
    SELECT
        z.location_id,
        z.borough,
        z.zone,
        w.weather_condition,
        w.weather_hour,
        COALESCE(c.trip_count, 0) AS trip_count
    FROM taxi_zones AS z
    CROSS JOIN relevant_weather_hours AS w
    LEFT JOIN zone_hour_counts AS c
        ON z.location_id = c.location_id
       AND w.weather_hour = c.pickup_hour
), zone_weather_average AS (
    SELECT
        location_id,
        borough,
        zone,
        weather_condition,
        AVG(trip_count) AS average_hourly_demand
    FROM zone_weather_hour_grid
    GROUP BY location_id, borough, zone, weather_condition
), variation AS (
    SELECT
        location_id,
        borough,
        zone,
        COUNT(*) AS weather_categories,
        MIN(average_hourly_demand) AS minimum_weather_demand,
        MAX(average_hourly_demand) AS maximum_weather_demand,
        MAX(average_hourly_demand) - MIN(average_hourly_demand) AS demand_range,
        STDDEV_POP(average_hourly_demand) AS demand_stddev
    FROM zone_weather_average
    GROUP BY location_id, borough, zone
    HAVING COUNT(*) >= 2
), ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY demand_range DESC) AS variation_rank
    FROM variation
)
SELECT
    variation_rank,
    location_id,
    borough,
    zone,
    weather_categories,
    ROUND(minimum_weather_demand, 3) AS minimum_average_hourly_demand,
    ROUND(maximum_weather_demand, 3) AS maximum_average_hourly_demand,
    ROUND(demand_range, 3) AS average_hourly_demand_range,
    ROUND(demand_stddev, 3) AS average_hourly_demand_stddev
FROM ranked
WHERE variation_rank <= 20
ORDER BY variation_rank, location_id
"""

RESULTS["zone_weather_demand_variation"] = run_query(
    "zone_weather_demand_variation",
    QUERIES["zone_weather_demand_variation"],
)

zone_weather_demand_variation execution time: 23.920 seconds
zone_weather_demand_variation result rows: 20
+--------------+-----------+---------+-------------------------+------------------+-----------------------------+-----------------------------+---------------------------+----------------------------+
|variation_rank|location_id|borough  |zone                     |weather_categories|minimum_average_hourly_demand|maximum_average_hourly_demand|average_hourly_demand_range|average_hourly_demand_stddev|
+--------------+-----------+---------+-------------------------+------------------+-----------------------------+-----------------------------+---------------------------+----------------------------+
|1             |237        |Manhattan|Upper East Side South    |5                 |189.492                      |260.709                      |71.217                     |27.997                      |
|2             |236        |Manhattan|Upper East Side North    |5                 |184.11

## Query 5: Peak travel hours for each day of the week

- Grain: day of week × pickup hour.
- Measure: trip count.
- `DENSE_RANK` retains all hours tied for the highest demand on each weekday.

In [7]:
QUERIES["peak_hours_by_day_of_week"] = """
WITH day_hour_demand AS (
    SELECT
        DAYOFWEEK(pickup_datetime) AS day_of_week_number,
        DATE_FORMAT(pickup_datetime, 'EEEE') AS day_of_week,
        HOUR(pickup_datetime) AS hour_of_day,
        COUNT(trip_id) AS trip_count
    FROM taxi_trips
    GROUP BY
        DAYOFWEEK(pickup_datetime),
        DATE_FORMAT(pickup_datetime, 'EEEE'),
        HOUR(pickup_datetime)
), ranked AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY day_of_week_number
            ORDER BY trip_count DESC
        ) AS demand_rank
    FROM day_hour_demand
)
SELECT
    day_of_week_number,
    day_of_week,
    hour_of_day AS peak_hour,
    trip_count
FROM ranked
WHERE demand_rank = 1
ORDER BY day_of_week_number, peak_hour
"""

RESULTS["peak_hours_by_day_of_week"] = run_query(
    "peak_hours_by_day_of_week", QUERIES["peak_hours_by_day_of_week"]
)

peak_hours_by_day_of_week execution time: 15.166 seconds
peak_hours_by_day_of_week result rows: 7
+------------------+-----------+---------+----------+
|day_of_week_number|day_of_week|peak_hour|trip_count|
+------------------+-----------+---------+----------+
|1                 |Sunday     |0        |79602     |
|2                 |Monday     |18       |80484     |
|3                 |Tuesday    |18       |99615     |
|4                 |Wednesday  |18       |110915    |
|5                 |Thursday   |18       |119125    |
|6                 |Friday     |18       |102923    |
|7                 |Saturday   |19       |96491     |
+------------------+-----------+---------+----------+



## Query 6: Monthly trends in taxi demand

- Grain: calendar month.
- Measures: monthly trip count, average daily demand, and month-over-month absolute and percentage changes.
- Average daily demand controls for differences in the number of represented days.

In [8]:
QUERIES["monthly_demand_trends"] = """
WITH monthly AS (
    SELECT
        pickup_year,
        pickup_month,
        MAKE_DATE(pickup_year, pickup_month, 1) AS month_start,
        COUNT(trip_id) AS trip_count,
        COUNT(DISTINCT pickup_date) AS observed_days,
        COUNT(trip_id) / COUNT(DISTINCT pickup_date) AS average_daily_demand
    FROM taxi_trips
    GROUP BY pickup_year, pickup_month
), with_previous AS (
    SELECT
        *,
        LAG(trip_count) OVER (ORDER BY month_start) AS previous_month_trip_count
    FROM monthly
)
SELECT
    pickup_year,
    pickup_month,
    month_start,
    trip_count,
    observed_days,
    ROUND(average_daily_demand, 2) AS average_daily_demand,
    trip_count - previous_month_trip_count AS month_over_month_change,
    ROUND(
        100.0 * (trip_count - previous_month_trip_count)
        / NULLIF(previous_month_trip_count, 0),
        2
    ) AS month_over_month_change_pct
FROM with_previous
ORDER BY month_start
"""

RESULTS["monthly_demand_trends"] = run_query(
    "monthly_demand_trends", QUERIES["monthly_demand_trends"]
)

monthly_demand_trends execution time: 9.002 seconds
monthly_demand_trends result rows: 8
+-----------+------------+-----------+----------+-------------+--------------------+-----------------------+---------------------------+
|pickup_year|pickup_month|month_start|trip_count|observed_days|average_daily_demand|month_over_month_change|month_over_month_change_pct|
+-----------+------------+-----------+----------+-------------+--------------------+-----------------------+---------------------------+
|2002       |12          |2002-12-01 |3         |1            |3.0                 |NULL                   |NULL                       |
|2008       |12          |2008-12-01 |1         |1            |1.0                 |-2                     |-66.67                     |
|2009       |1           |2009-01-01 |4         |1            |4.0                 |3                      |300.00                     |
|2023       |12          |2023-12-01 |10        |1            |10.0                |6    